![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/annotation-merger/Merging_Annotation_Columns.ipynb)

# **Merging Annotation Columns with MultiColumnAssembler**

## Overview

When working with Spark NLP pipelines, you may encounter situations where multiple annotators produce separate annotation columns. For example, `ReaderAssembler` splits document content into `document_text` and `document_table` columns, but a downstream annotator like `AutoGGUFVisionModel` expects a **single** input column.

The `MultiColumnAssembler` solves this by combining multiple annotation columns into one. It:

- Concatenates annotations from all input columns into a single output column
- Preserves the original metadata of each annotation
- Adds a `source_column` metadata key so you can trace which column an annotation came from
- Allows changing the output annotator type (default: `document`)

In this notebook we walk through several examples demonstrating these capabilities.

## 0. Colab Setup

In [ ]:
!pip install -q pyspark spark-nlp

In [ ]:
import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

spark = sparknlp.start()

print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)

## 1. Merging Two Document Columns

The simplest use case: two `DocumentAssembler` stages produce separate columns, and we merge them into one.

In [4]:
data = spark.createDataFrame(
    [("Spark NLP is an open-source NLP library.", "Name | Age\nAlice | 30\nBob | 25")],
    ["text", "table"]
)

doc_text = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document_text")

doc_table = DocumentAssembler() \
    .setInputCol("table") \
    .setOutputCol("document_table")

merger = MultiColumnAssembler() \
    .setInputCols(["document_text", "document_table"]) \
    .setOutputCol("merged_document")

pipeline = Pipeline().setStages([doc_text, doc_table, merger]).fit(data)
result = pipeline.transform(data)

result.selectExpr("merged_document.result").show(truncate=False)

+----------------------------------------------------------------------------+
|result                                                                      |
+----------------------------------------------------------------------------+
|[Spark NLP is an open-source NLP library., Name | Age\nAlice | 30\nBob | 25]|
+----------------------------------------------------------------------------+



We now have both text and table content in a single `merged_document` column that can be fed into any downstream annotator.

## 2. Inspecting Metadata

Every merged annotation automatically gets a `source_column` key in its metadata, so you always know where each annotation originally came from.

In [5]:
data = spark.createDataFrame(
    [("Hello world", "Goodbye world")],
    ["text1", "text2"]
)

da1 = DocumentAssembler().setInputCol("text1").setOutputCol("doc1")
da2 = DocumentAssembler().setInputCol("text2").setOutputCol("doc2")

merger = MultiColumnAssembler() \
    .setInputCols(["doc1", "doc2"]) \
    .setOutputCol("merged")

pipeline = Pipeline().setStages([da1, da2, merger]).fit(data)
result = pipeline.transform(data)

# Show full annotation structure including metadata
result.selectExpr("explode(merged) as annotation").selectExpr(
    "annotation.result",
    "annotation.metadata.source_column"
).show(truncate=False)

+-------------+-------------+
|result       |source_column|
+-------------+-------------+
|Hello world  |doc1         |
|Goodbye world|doc2         |
+-------------+-------------+



## 3. Merging Three or More Columns

`MultiColumnAssembler` supports any number of input columns, not just two.

In [6]:
ata = spark.createDataFrame(
    [("First paragraph.", "Second paragraph.", "Third paragraph.")],
    ["text1", "text2", "text3"]
)

da1 = DocumentAssembler().setInputCol("text1").setOutputCol("doc1")
da2 = DocumentAssembler().setInputCol("text2").setOutputCol("doc2")
da3 = DocumentAssembler().setInputCol("text3").setOutputCol("doc3")

merger = MultiColumnAssembler() \
    .setInputCols(["doc1", "doc2", "doc3"]) \
    .setOutputCol("merged")

pipeline = Pipeline().setStages([da1, da2, da3, merger]).fit(data)
result = pipeline.transform(data)

result.selectExpr("merged.result").show(truncate=False)

+-------------------------------------------------------+
|result                                                 |
+-------------------------------------------------------+
|[First paragraph., Second paragraph., Third paragraph.]|
+-------------------------------------------------------+



## 4. Changing the Output Annotator Type

The output annotator type defaults to `document`, but you can change it to match what a downstream annotator expects (e.g., `chunk`).

In [8]:
from sparknlp.base import DocumentAssembler, MultiColumnAssembler
from pyspark.ml import Pipeline

data = spark.createDataFrame(
    [("Entity one", "Entity two")],
    ["text1", "text2"]
)

da1 = DocumentAssembler().setInputCol("text1").setOutputCol("doc1")
da2 = DocumentAssembler().setInputCol("text2").setOutputCol("doc2")

merger = MultiColumnAssembler() \
    .setInputCols(["doc1", "doc2"]) \
    .setOutputCol("merged_chunks") \
    .setOutputAsAnnotatorType("chunk")

pipeline = Pipeline().setStages([da1, da2, merger]).fit(data)
result = pipeline.transform(data)

# Verify the annotator type is now 'chunk'
result.selectExpr(
    "explode(merged_chunks) as annotation"
).selectExpr(
    "annotation.annotatorType",
    "annotation.result"
).show(truncate=False)

+-------------+----------+
|annotatorType|result    |
+-------------+----------+
|chunk        |Entity one|
|chunk        |Entity two|
+-------------+----------+



## 5. Working with Multiple Rows

`MultiColumnAssembler` processes each row independently, so it scales naturally across DataFrames with many rows.

In [9]:
from sparknlp.base import DocumentAssembler, MultiColumnAssembler
from pyspark.ml import Pipeline

data = spark.createDataFrame(
    [
        ("Row 1 text content", "Row 1 table content"),
        ("Row 2 text content", "Row 2 table content"),
        ("Row 3 text content", "Row 3 table content"),
    ],
    ["text", "table"]
)

da1 = DocumentAssembler().setInputCol("text").setOutputCol("doc_text")
da2 = DocumentAssembler().setInputCol("table").setOutputCol("doc_table")

merger = MultiColumnAssembler() \
    .setInputCols(["doc_text", "doc_table"]) \
    .setOutputCol("merged")

pipeline = Pipeline().setStages([da1, da2, merger]).fit(data)
result = pipeline.transform(data)

result.selectExpr("merged.result").show(truncate=False)

+-----------------------------------------+
|result                                   |
+-----------------------------------------+
|[Row 1 text content, Row 1 table content]|
|[Row 2 text content, Row 2 table content]|
|[Row 3 text content, Row 3 table content]|
+-----------------------------------------+

